# BreastDCEDL: exploratory data analysis

What this notebook shows, top to bottom:

- **Part A (metadata, runs anywhere with the repo):**
  - cohorts, labels and splits;
  - missing values;
  - clinical features;
  - imaging parameters and DCE phases;
  - tumour size;
  - how each feature relates to pCR.
- **Part B (images, needs a slice cache):**
  - cache quality checks;
  - the three DCE phases of real patients, shown separately and as the RGB image the model sees;
  - enhancement curves.

Part B runs if it finds a cache:
- on Kaggle with `mnj-cache-*` attached;
- locally if you copy `cache_index.csv`, `cache_meta.json` and a few `<pid>.npz` files into
  `data/cache/<cohort>/`.

**Test-set rule:** split *counts* are shown for all splits (they are published). Anything that relates
features or images to the pCR label uses **training patients only**, so we never learn anything from test
labels.

In [ ]:
import glob, json, os, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

# repo root: the folder that holds data/metadata (walk up from here), else a Kaggle clone
cands = [Path.cwd(), *Path.cwd().parents, Path("/tmp/repo"), Path("/kaggle/working/MRI_chemo_response")]
ROOT = next(p for p in cands if (p / "data" / "metadata").is_dir())
sys.path.insert(0, str(ROOT))
from src.data.metadata import load_metadata, load_split
from src.data.clinical import race_column

pd.set_option("display.width", 160, "display.max_columns", 40)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
COHORTS = ["ispy1", "ispy2", "duke"]
COLORS = {"ispy1": "#2a6fdb", "ispy2": "#e07b00", "duke": "#1b9e5a"}

meta = load_metadata()                     # checks the CSV md5, adds cohort / split / bbox volume
split = load_split()
df = meta.drop(columns=["split"]).join(split[["split"]])
df["race"] = race_column(df)
df["subtype"] = df["HR_HER2_STATUS"].map({"HRposHER2neg": "HR+/HER2-", "HER2pos": "HER2+", "TripleNeg": "triple-neg"})
lab = df[df.labeled]
train = lab[lab.split == "train"]            # the only labels we analyse
print(f"repo: {ROOT}\n{len(df)} patients, {len(lab)} labeled, {len(train)} labeled train patients")

## A1. Cohorts, labels and splits

- **Labeled** means the patient has a pCR outcome.
- **pCR = 1:** no invasive cancer left at surgery after neoadjuvant chemotherapy.
- Every I-SPY patient is labeled; only 298 of 916 Duke patients are.

In [ ]:
t = df.groupby("cohort").agg(patients=("pid", "size"), labeled=("labeled", "sum"),
                             pCR_pos=("pCR", "sum")).loc[COHORTS]
t["pCR_neg"] = t.labeled - t.pCR_pos
t["pCR_rate"] = (t.pCR_pos / t.labeled).round(3)
t.loc["total"] = t.sum(numeric_only=True)
t.loc["total", "pCR_rate"] = round(t.loc["total", "pCR_pos"] / t.loc["total", "labeled"], 3)
display(t.astype({c: int for c in ["patients", "labeled", "pCR_pos", "pCR_neg"]}))

n = pd.crosstab(lab.cohort, lab.split).loc[COHORTS, ["train", "val", "test"]]
p = lab.pivot_table(index="cohort", columns="split", values="pCR", aggfunc="sum").loc[COHORTS, ["train", "val", "test"]].astype(int)
print("labeled patients per split, 'n (pCR+)':")
display(n.astype(str) + " (" + p.astype(str) + ")")
print("unlabeled Duke patients per split (excluded from supervised training and all metrics):")
display(df[~df.labeled].groupby("split").size().rename("unlabeled").to_frame().T)

## A2. Missing values

How the models handle them (`src/data/clinical.py`, fitted on train only):
- **Categorical features** (HR, HER2, race, menopause) get an extra **"missing"** category.
- **Continuous features** (age, bbox volume) get the **train median** plus a "was missing" flag.

Note that menopause is missing for **every** I-SPY1 patient.

In [ ]:
cols = ["pCR", "HR", "HER2", "age", "menopause", "race", "tum_vol", "log_bbox_vol"]
miss = df.groupby("cohort")[cols].apply(lambda g: g.isna().sum()).loc[COHORTS]
miss.loc["total"] = miss.sum()
display(miss)

## A3. Clinical features by cohort (all labeled patients)

- **HR+:** the tumour has oestrogen and/or progesterone receptors.
- **HER2+:** the tumour over-expresses HER2.
- **Subtype:** HR+/HER2−, HER2+ (any HR), or triple-negative.
- The receptor thresholds differ by cohort:
  - HR+: ≥1% stained cells in I-SPY2, ≥10% in I-SPY1, Allred >3 in Duke;
  - HER2+: FISH ratio ≥2.0 in I-SPY, >2.2 in Duke.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.3))
for c in COHORTS:
    axes[0].hist(lab[lab.cohort == c].age.dropna(), bins=25, alpha=0.5, color=COLORS[c], label=c)
axes[0].set(title="age (years)", xlabel="age"); axes[0].legend()
for ax, col, title in [(axes[1], "subtype", "subtype"), (axes[2], "race", "race"), (axes[3], "menopause", "menopause (1 = post)")]:
    ct = pd.crosstab(lab.cohort, lab[col].fillna("missing").astype(str), normalize="index").loc[COHORTS]
    ct.plot.bar(stacked=True, ax=ax, rot=0, colormap="tab20c", legend=True)
    ax.set(title=title, ylabel="fraction of labeled patients"); ax.legend(fontsize=7, loc="lower right")
plt.tight_layout(); plt.show()
display(lab.groupby("cohort")[["age", "HR", "HER2"]].mean().loc[COHORTS].round(2).rename(columns={"HR": "HR+ rate", "HER2": "HER2+ rate"}))

## A4. Imaging parameters and DCE phases

- **`n_xy`:** in-plane size of the *original* scan. The archive volumes are 256×256 windows of it, which is
  why the metadata tumour boxes had to be shifted (`challenges.md`).
- **`n_z`:** number of slices.
- **`slice_thick` / `xy_spacing`:** mm between slices and mm per pixel.
- **`n_times`:** number of DCE acquisitions.
- **`pre` / `post_early` / `post_late`:** which acquisition is used for the R, G and B channels.

In [ ]:
for col in ["n_xy", "n_times", "pre", "post_early", "post_late"]:
    print(f"{col}:", {c: df[df.cohort == c][col].value_counts().sort_index().astype(int).to_dict() for c in COHORTS})
fig, axes = plt.subplots(1, 3, figsize=(14, 3.2))
for ax, col, unit in zip(axes, ["n_z", "slice_thick", "xy_spacing"], ["slices", "mm", "mm / pixel"]):
    ax.boxplot([df[df.cohort == c][col].dropna() for c in COHORTS], showfliers=False)
    ax.set_xticks([1, 2, 3], COHORTS); ax.set(title=col, ylabel=unit)
plt.tight_layout(); plt.show()

## A5. Tumour size

- **bbox volume** = (slices spanned × slice thickness) × (rows × pixel size) × (columns × pixel size), in mm³,
  from the metadata box.
- We use its **log** because the raw volume is very skewed.
- `tum_vol` is the dataset's own tumour volume, from the mask voxel count in I-SPY. For I-SPY you can see how
  well the bbox volume tracks it.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.4))
for c in COHORTS:
    d = df[df.cohort == c]
    axes[0].hist(d.bbox_vol_mm3 / 1000, bins=40, range=(0, 150), alpha=0.5, color=COLORS[c], label=c)
    axes[1].hist(d.log_bbox_vol, bins=30, alpha=0.5, color=COLORS[c], label=c)
axes[0].set(title="bbox volume (cm³), raw: skewed", xlabel="cm³"); axes[0].legend()
axes[1].set(title="log bbox volume: roughly symmetric", xlabel="log(mm³)")
isp = df[df.cohort != "duke"].dropna(subset=["tum_vol"])
axes[2].scatter(np.log(isp.tum_vol * 1000), isp.log_bbox_vol, s=4, alpha=0.4)
r = np.corrcoef(np.log(isp.tum_vol * 1000), isp.log_bbox_vol)[0, 1]
axes[2].set(title=f"I-SPY: log tum_vol vs log bbox vol (r = {r:.2f})", xlabel="log tum_vol (assumed cm³ → mm³)", ylabel="log bbox vol")
plt.tight_layout(); plt.show()

## A6. What predicts pCR? (training patients only)

- **pCR rate by subtype and cohort:** expect HR+/HER2− lowest and HER2+ / triple-negative highest.
- **Univariate AUC:** each feature's AUC on its own; 0.5 = no signal.
  - Values below 0.5 mean that **higher values go with less pCR** (e.g. larger tumours).
  - The table also shows `max(AUC, 1 − AUC)`, the signal strength regardless of direction.

In [ ]:
rate = train.pivot_table(index="subtype", columns="cohort", values="pCR", aggfunc=["mean", "size"])
display(rate.round(2))

rows = []
feats = {"HR": train.HR, "HER2": train.HER2, "age": train.age, "menopause": train.menopause,
         "race_black": (train.race == "black").astype(float).where(train.race.notna()),
         "log_bbox_vol": train.log_bbox_vol}
for name, x in feats.items():
    for c in ["all", *COHORTS]:
        m = (train.cohort == c) if c != "all" else pd.Series(True, index=train.index)
        ok = m & x.notna()
        y = train.pCR[ok]
        if y.nunique() == 2 and x[ok].nunique() > 1:
            a = roc_auc_score(y, x[ok])
            rows.append({"feature": name, "cohort": c, "n": int(ok.sum()), "AUC": round(a, 3), "strength": round(max(a, 1 - a), 3)})
uni = pd.DataFrame(rows).pivot(index="feature", columns="cohort", values="AUC")[["all", *COHORTS]]
display(uni)

fig, ax = plt.subplots(figsize=(6, 3.2))
for i, c in enumerate(COHORTS):
    d = train[train.cohort == c]
    ax.boxplot([d[d.pCR == 0].log_bbox_vol, d[d.pCR == 1].log_bbox_vol], positions=[3 * i, 3 * i + 1], showfliers=False)
ax.set_xticks([0, 1, 3, 4, 6, 7], [f"{c}\n{t}" for c in COHORTS for t in ("no pCR", "pCR")], fontsize=8)
ax.set(title="tumour size vs pCR (train)", ylabel="log bbox volume"); plt.tight_layout(); plt.show()

### What Part A means for modelling

- **Class imbalance:** about 30% pCR+, so AUPRC's chance level is about 0.30, not 0.5. That's also why the
  neural models use focal loss.
- **The clinical signal is concentrated in HR and HER2** (subtype). That's why the clinical-only baseline B1
  is a hard bar.
- **Cohorts differ** in pCR rate, receptor definitions, slice thickness and missingness. Report results per
  cohort, and watch for "cohort shortcuts": for example, `menopause = missing` ≈ I-SPY1.
- **Duke's 618 unlabeled patients** are only usable for self-supervised pretraining (train + val only).

# Part B: images (needs a slice cache)

Each cache file `<pid>.npz` holds:
- `rgb`: (slices, 240, 240, 3) uint8, R/G/B = pre/early/late, jointly clipped at the 0.5–99.5 percentiles;
- `t0:t1`: the tumour slices;
- `bbox`: the tumour box inside the crop;
- `mask`: the tumour mask, I-SPY only.

The model sees a 224×224 crop of this, normalised with ImageNet mean and std.

In [ ]:
roots = [Path(os.environ["EDA_CACHE_ROOT"])] if os.environ.get("EDA_CACHE_ROOT") else []
roots += [ROOT / "data" / "cache", Path("/kaggle/input"), Path("/kaggle/working")]
CACHES = {}
for r in roots:
    for depth in range(6):
        for f in glob.glob(os.path.join(r, *["*"] * depth, "cache_meta.json")):
            CACHES.setdefault(json.load(open(f))["cohort"], Path(f).parent)
HAVE_CACHE = bool(CACHES)
print("caches found:", {c: str(p) for c, p in CACHES.items()} or "none -> Part B skipped")

## B1. Cache quality checks

In [ ]:
if HAVE_CACHE:
    for c, d in CACHES.items():
        idx = pd.read_csv(d / "cache_index.csv")
        print(f"\n== {c}: {len(idx)} patients")
        for col in ["status", "z_source", "xy_source"]:
            if col in idx:
                print(f"  {col}:", idx[col].value_counts(dropna=False).to_dict())
        ok = idx[idx.status == "ok"]
        if {"t0", "t1"} <= set(ok.columns):
            print("  tumour slices per patient:", ok.eval("t1 - t0").describe()[["min", "50%", "max"]].to_dict())
        if "mask_csv_iou_yx" in ok:
            iou = ok.mask_csv_iou_yx.dropna()
            if len(iou):
                print("  CSV box vs mask IoU: min %.2f, median %.2f" % (iou.min(), iou.median()))
        err = idx[idx.status != "ok"]
        if len(err):
            display(err[[c for c in ["pid", "status", "note"] if c in err]].head(10))

## B2. One patient, phase by phase

For a few **training** patients (pCR+ and pCR−) we show, from the middle tumour slice:
- each DCE phase on its own (grey);
- the RGB composite;
- the tumour box (yellow) and, for I-SPY, the mask outline (cyan).

The tumour should be **dark in pre, bright in early**, and then either fade (washout) or stay bright in late.

In [ ]:
def load(pid, cohort):
    p = CACHES[cohort] / f"{pid}.npz"
    return np.load(p) if p.exists() else None

def show_patient(pid, cohort):
    d = load(pid, cohort)
    if d is None:
        return False
    k = (int(d["t0"]) + int(d["t1"])) // 2
    img = d["rgb"][k]
    y0, y1, x0, x1 = d["bbox"]
    fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
    for i, name in enumerate(["R = pre-contrast", "G = early post", "B = late post"]):
        axes[i].imshow(img[..., i], cmap="gray", vmin=0, vmax=255); axes[i].set_title(name)
    axes[3].imshow(img); axes[3].set_title("RGB (model input)")
    for ax in axes:
        ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec="yellow", lw=1))
        if "mask" in d and d["mask"][k].any():
            ax.contour(d["mask"][k], levels=[0.5], colors="cyan", linewidths=0.7)
        ax.axis("off")
    lab_ = int(train.loc[pid, "pCR"])
    fig.suptitle(f"{pid} ({cohort}) | pCR = {lab_} | {train.loc[pid, 'subtype']} | slice {k} of tumour slices {int(d['t0'])}-{int(d['t1'])}")
    plt.tight_layout(); plt.show()
    return True

if HAVE_CACHE:
    rng = np.random.default_rng(0)
    for c in CACHES:
        avail = {Path(f).stem for f in glob.glob(str(CACHES[c] / "*.npz"))}
        for y in (1, 0):
            pids = [p for p in train[(train.cohort == c) & (train.pCR == y)].pid if p in avail]
            for pid in rng.permutation(pids)[:1]:
                show_patient(pid, c)

## B3. Enhancement curves: does the image carry pCR signal?

- For up to 150 training patients per cohort, take the mean intensity of each phase inside the tumour (the
  mask for I-SPY, the box for Duke).
- Then compare pCR+ with pCR− patients, as **enhancement relative to pre-contrast**.
- Overlapping curves mean the simple kinetics differ little between responders and non-responders. Any image
  model then has to find subtler patterns: texture, shape, heterogeneity.

In [ ]:
if HAVE_CACHE:
    fig, axes = plt.subplots(1, len(CACHES), figsize=(4.5 * len(CACHES), 3.3), squeeze=False)
    for ax, c in zip(axes[0], CACHES):
        avail = {Path(f).stem for f in glob.glob(str(CACHES[c] / "*.npz"))}
        sub = train[(train.cohort == c) & train.pid.isin(avail)]
        sub = sub.sample(min(150, len(sub)), random_state=0)
        curves = {0: [], 1: []}
        for pid, y in zip(sub.pid, sub.pCR.astype(int)):
            d = load(pid, c)
            t0, t1 = int(d["t0"]), int(d["t1"])
            vol = d["rgb"][t0:t1].astype(float)
            if "mask" in d and d["mask"][t0:t1].any():
                sel = d["mask"][t0:t1] > 0
            else:
                y0, y1, x0, x1 = d["bbox"]
                sel = np.zeros(vol.shape[:3], bool); sel[:, y0:y1, x0:x1] = True
            m = vol[sel].mean(axis=0)            # mean R, G, B inside the tumour
            curves[y].append(m / max(m[0], 1.0))  # relative to pre-contrast
        for y, style in [(0, "-"), (1, "--")]:
            a = np.array(curves[y])
            if len(a):
                ax.errorbar([0, 1, 2], a.mean(0), yerr=a.std(0) / np.sqrt(len(a)), ls=style, marker="o", capsize=3,
                            label=f"pCR={y} (n={len(a)})")
        ax.set(xticks=[0, 1, 2], xticklabels=["pre", "early", "late"], title=c, ylabel="intensity / pre")
        ax.legend()
    plt.tight_layout(); plt.show()

## B4. What the model actually receives

- The centre 224×224 crop of three tumour slices, as the model sees them: after cropping, before ImageNet
  normalisation.
- During training it also gets random flips, ±15° rotation, 0.9–1.1 scaling, ±10% intensity and ±8 px
  shifts.
- Check that the tumour (yellow box) is inside the crop and the colours look the same as in B2.

In [ ]:
if HAVE_CACHE:
    c = next(iter(CACHES))
    avail = sorted(Path(f).stem for f in glob.glob(str(CACHES[c] / "*.npz")))
    pid = next(p for p in avail if p in train.index)
    d = load(pid, c)
    t0, t1 = int(d["t0"]), int(d["t1"])
    ks = np.unique(np.linspace(t0, t1 - 1, 3).round().astype(int))
    off = (240 - 224) // 2
    y0, y1, x0, x1 = d["bbox"] - off
    fig, axes = plt.subplots(1, len(ks), figsize=(3.4 * len(ks), 3.4), squeeze=False)
    for ax, k in zip(axes[0], ks):
        ax.imshow(d["rgb"][k, off:off + 224, off:off + 224])
        ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec="yellow", lw=1))
        ax.set_title(f"{pid} slice {k}"); ax.axis("off")
    plt.tight_layout(); plt.show()